# Trader knowledge base: your own free AI server in Google Colab

This notebook runs an open AI model on a free Colab GPU and gives you a public address. Paste that address into the website (**API keys > Optional: use your own free GPU**) and the knowledge base uses your Colab instead of Groq, so there are no per-minute or daily limits.

**Steps**
1. Menu **Runtime > Change runtime type > T4 GPU**, then Save.
2. Menu **Runtime > Run all**. Wait about 5 to 10 minutes the first time (it downloads the model).
3. The last cell prints an address like `https://something.trycloudflare.com` and a model name. Paste both into the website and press **Test Colab connection**.
4. Keep this tab open and the last cell running while videos are processed.

**Good to know**
- A free Colab GPU is slower than Groq and the model is smaller (7B instead of 120B), so extracted rules can be less complete. The website still checks every quote against the transcript.
- The address changes every time you run the notebook. Anyone who has the address can use your GPU, so do not share it.
- Free Colab can disconnect after idle time or about 12 hours. Run all again and paste the new address. Nothing is lost: press **Retry** on the website.
- Instagram reels still use Groq Whisper through your Make scenario. Only translating, extracting and checking rules move to Colab.
- Google's Colab rules apply. Use your own account, for your own work.

In [ ]:
# 1. Check the GPU and choose the model
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: use Runtime > Change runtime type > T4 GPU")
MODEL = "qwen2.5:7b"      # fits the free T4 and is fast. A stronger but slower choice: "qwen2.5:14b"

In [ ]:
# 2. Install Ollama (the model server)
import subprocess
subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True, check=True)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Start the server and download the model (first time: a few minutes)
import os, subprocess, time, requests, json
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_ORIGINS="*", OLLAMA_NUM_PARALLEL="2",
           OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try: requests.get("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception: time.sleep(1)
else:
    raise SystemExit("Ollama did not start. Look at /content/ollama.log")
subprocess.run(["ollama", "pull", MODEL], env=env, check=True)

# quick self-test: the website asks for JSON replies
r = requests.post("http://127.0.0.1:11434/v1/chat/completions", json={
    "model": MODEL, "temperature": 0, "response_format": {"type": "json_object"},
    "messages": [{"role": "user", "content": 'Reply with JSON {"ok": true} and nothing else.'}]}, timeout=300)
print("Self-test reply:", r.json()["choices"][0]["message"]["content"])

In [ ]:
# 4. Open a public address and keep running (leave this cell running while you use the website)
import os, re, subprocess, threading, time, requests
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "127.0.0.1:11434", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, started = None, time.time()
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
    if time.time() - started > 90: break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep the pipe drained
if not url: raise SystemExit("Could not open the public address. Run this cell again.")
time.sleep(3)
print("=" * 60)
print("Paste these two into the website (API keys > Colab):")
print("  Colab address:", url)
print("  Model name:   ", MODEL)
print("=" * 60)
print("Keep this cell running. It prints a dot every minute.")
while tunnel.poll() is None:
    time.sleep(60); print(".", end="", flush=True)
print("\nThe tunnel stopped. Run this cell again and paste the new address.")